<a href="https://colab.research.google.com/github/AlfCR/dnn-estadia/blob/main/dnn_estadia_ACP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# Clonación del repositorio
!git clone https://github.com/AlfCR/dnn-estadia.git

# Verificar la conexión al repositorio
%cd dnn-estadia

Cloning into 'dnn-estadia'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 6 (delta 0), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), 4.84 KiB | 619.00 KiB/s, done.
/content/dnn-estadia/dnn-estadia


In [20]:
!pwd
!git remote -v
!git log --oneline

/content/dnn-estadia/dnn-estadia
origin	https://github.com/AlfCR/dnn-estadia.git (fetch)
origin	https://github.com/AlfCR/dnn-estadia.git (push)
0cc6d1b (HEAD -> main, origin/main, origin/HEAD) E2: Entorno de desarrollo configurado y validado con acceso a los datasets.
b0fd2bb Prueba de conexion Colab-GitHub


In [21]:
#Verificación de disponibilidad y versión  de TensorFlow
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

2.20.0
[]


In [22]:
#Verificación de  disponibilidad y versión  PyTorch
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cpu
False


In [23]:
#Acceso a los dataset
import pandas as pd

ruta = '/content/drive/MyDrive/Colab Notebooks/dataset/Mediciones Agosto 2024 Chiapas/torqueTrackLog.csv'
df = pd.read_csv(ruta)
print(f"Filas: {len(df)}, Columnas: {len(df.columns)}")
df.head()

Filas: 850, Columnas: 13


,GPS Time,Device Time,Longitude,Latitude,GPS Speed (Meters/second),Horizontal Dilution of Precision,Altitude,Bearing,G(x),G(y),G(z),G(calibrated),Unnamed: 12
0,Wed Aug 21 08:16:07 CST 2024,21-ago-2024 08:16:07.301,-93.146422,16.742990,0.046325,1.6,653.5,134.00,-0.67005,8.179951,5.68800,0.037802,NaN
1,Wed Aug 21 08:16:08 CST 2024,21-ago-2024 08:16:08.213,-93.146423,16.742990,0.045810,1.6,653.3,161.58,-0.66705,8.188050,5.64705,0.035194,NaN
2,Wed Aug 21 08:16:09 CST 2024,21-ago-2024 08:16:09.213,-93.146422,16.742990,0.033457,1.6,653.0,245.70,-0.31605,7.897950,6.33000,0.013965,NaN
3,Wed Aug 21 08:16:10 CST 2024,21-ago-2024 08:16:10.213,-93.146427,16.742992,0.123533,1.5,653.0,336.02,-2.82195,7.041000,7.41000,0.074501,NaN
4,Wed Aug 21 08:16:11 CST 2024,21-ago-2024 08:16:11.213,-93.146425,16.742990,0.091106,1.5,652.9,105.44,-2.35305,5.194950,6.42300,0.044035,NaN


Analizar la cantidad de archivos que contamos en el proyecto

In [24]:
import os

dataset_directory = '/content/drive/MyDrive/Colab Notebooks/dataset/Mediciones Agosto 2024 Chiapas/'
# Comprobar si el directorio existe antes de listar
if os.path.exists(dataset_directory):
    # Listar todos los elementos que sean archivos (omitir carpetas)
    all_files = [f for f in os.listdir(dataset_directory)
                 if os.path.isfile(os.path.join(dataset_directory, f))]

    if not all_files:
        print(f"No se encontraron archivos en '{dataset_directory}'")
    else:
        print(f"Se encontraron {len(all_files)} archivos de todo tipo:")
        for f in all_files:
            print(f"- {f}")
else:
    print(f"La ruta '{dataset_directory}' no existe.")

Se encontraron 6 archivos de todo tipo:
- torqueTrackLog(2).csv
- torqueTrackLog(3).csv
- torqueTrackLog(4).csv
- torqueTrackLog(1).csv
- torqueTrackLog(5).csv
- torqueTrackLog.csv


In [25]:
# Segundo dataset
import os

dataset_directory = '/content/drive/MyDrive/Colab Notebooks/dataset/Measurement complementary'

# Comprobar si el directorio existe antes de listar
if os.path.exists(dataset_directory):
    # Listar todos los elementos que sean archivos (omitir carpetas)
    all_files = [f for f in os.listdir(dataset_directory)
                 if os.path.isfile(os.path.join(dataset_directory, f))]

    if not all_files:
        print(f"No se encontraron archivos en '{dataset_directory}'")
    else:
        print(f"Se encontraron {len(all_files)} archivos de todo tipo:")
        for f in all_files:
            print(f"- {f}")
else:
    print(f"La ruta '{dataset_directory}' no existe.")

Se encontraron 11 archivos de todo tipo:
- trackLog-2022-nov.-10_09-23-59 (1).csv
- trackLog-2022-nov.-11_12-47-43.csv
- trackLog-2022-nov.-11_14-46-26.csv
- trackLog-2022-nov-12_20-05-07.csv
- trackLog-2022-nov-13_00-03-52.csv
- trackLog-2022-nov-14_09-03-16.csv
- trackLog-2022-nov-14_13-01-38.csv
- trackLog-2022-nov-16_09-32-26.csv
- trackLog-2022-nov-16_09-44-58.csv
- trackLog-2022-nov.-27_12-43-04.csv
- trackLog-2022-nov-16_09-44-58.xlsx


***Inventariar columnas de cada archivo***

In [26]:
import os
import pandas as pd

ruta1 = '/content/drive/MyDrive/Colab Notebooks/dataset/Measurement complementary'
ruta2 = '/content/drive/MyDrive/Colab Notebooks/dataset/Mediciones Agosto 2024 Chiapas'

def invetariar_columnas(carpeta):
  all_files = [f for f in os.listdir(carpeta) if os.path.isfile(os.path.join(carpeta, f))]
  inventario = {}
  for nombre_archivo in all_files:
      ruta_archivo = os.path.join(carpeta, nombre_archivo)

      try:
          df_headers = pd.read_csv(ruta_archivo, nrows=0)
          inventario[nombre_archivo] = df_headers.columns.tolist()
      except Exception as e:
          print(f"El archivo no se pudo leer {nombre_archivo}: {e}")

  return inventario

inventario1 = invetariar_columnas(ruta1)
inventario2 = invetariar_columnas(ruta2)

# Cantidad de columnas en cada archivo
print("Primera carpeta: ")
for nombre, columnas in inventario1.items():
    print(f"{nombre}: {len(columnas)} columnas")

print("\nSegunda carpeta: ")
for nombre, columnas in inventario2.items():
    print(f"{nombre}: {len(columnas)} columnas")

El archivo no se pudo leer trackLog-2022-nov-16_09-44-58.xlsx: 'utf-8' codec can't decode byte 0x9d in position 54: invalid start byte
Primera carpeta: 
trackLog-2022-nov.-10_09-23-59 (1).csv: 90 columnas
trackLog-2022-nov.-11_12-47-43.csv: 52 columnas
trackLog-2022-nov.-11_14-46-26.csv: 206 columnas
trackLog-2022-nov-12_20-05-07.csv: 206 columnas
trackLog-2022-nov-13_00-03-52.csv: 206 columnas
trackLog-2022-nov-14_09-03-16.csv: 206 columnas
trackLog-2022-nov-14_13-01-38.csv: 99 columnas
trackLog-2022-nov-16_09-32-26.csv: 99 columnas
trackLog-2022-nov-16_09-44-58.csv: 99 columnas
trackLog-2022-nov.-27_12-43-04.csv: 32 columnas

Segunda carpeta: 
torqueTrackLog(2).csv: 13 columnas
torqueTrackLog(3).csv: 13 columnas
torqueTrackLog(4).csv: 13 columnas
torqueTrackLog(1).csv: 13 columnas
torqueTrackLog(5).csv: 13 columnas
torqueTrackLog.csv: 13 columnas


In [27]:
import pandas as pd

# 1. Sacamos solo los archivos que SÍ se pudieron leer (evita el .xlsx problemático)
listas_de_columnas = list(inventario1.values())


# 2. Convertimos cada lista de columnas en un set (conjunto sin orden ni duplicados)
conjuntos_de_columnas = [set(columnas) for columnas in listas_de_columnas]

# 3. Intersección: columnas que aparecen en TODOS los archivos a la vez
interseccion = set.intersection(*conjuntos_de_columnas)

# 4. Unión: todas las columnas distintas que existen en al menos un archivo
union = set.union(*conjuntos_de_columnas)

print(f"Columnas en la intersección (comunes a todos): {len(interseccion)}")
print(f"Columnas en la unión (el total combinado): {len(union)}")

#Imprimir nombres de las columnas
print("Columnas en la intersección (comunes a todos):")
for columna in sorted(interseccion):
    print(f"- {columna}")


Columnas en la intersección (comunes a todos): 20
Columnas en la unión (el total combinado): 212
Columnas en la intersección (comunes a todos):
-  Altitude
-  Bearing
-  Device Time
-  G(calibrated)
-  G(x)
-  G(y)
-  G(z)
-  Horizontal Dilution of Precision
-  Latitude
-  Longitude
- Acceleration Sensor(Total)(g)
- Acceleration Sensor(Y axis)(g)
- Acceleration Sensor(Z axis)(g)
- GPS Latitude(°)
- GPS Longitude(°)
- GPS Speed (Meters/second)
- GPS Time
- Horsepower (At the wheels)(hp)
- Trip Time(Since journey start)(s)
- Voltage (OBD Adapter)(V)


### Limpieza de nombres de columna para 'Measurement complementary' (ruta1)

Limpiemos los nombres de las columnas eliminando los espacios en blanco al principio o al final. Esto ayudará a identificar las columnas que son realmente comunes y a evitar problemas con duplicados aparentes.

In [32]:
import os
import pandas as pd

ruta1 = '/content/drive/MyDrive/Colab Notebooks/dataset/Measurement complementary'
ruta2 = '/content/drive/MyDrive/Colab Notebooks/dataset/Mediciones Agosto 2024 Chiapas'

def invetariar_columnas(carpeta):
    all_files = [f for f in os.listdir(carpeta) if os.path.isfile(os.path.join(carpeta, f))]
    inventario = {}
    for nombre_archivo in all_files:
        ruta_archivo = os.path.join(carpeta, nombre_archivo)
        try:
            df_headers = pd.read_csv(ruta_archivo, nrows=0)
            columnas_limpias = [columna.strip() for columna in df_headers.columns]
            inventario[nombre_archivo] = columnas_limpias
        except Exception as e:
            print(f"El archivo no se pudo leer {nombre_archivo}: {e}")
    return inventario

# Generar inventarios de ambas carpetas
inventario1 = invetariar_columnas(ruta1)
inventario2 = invetariar_columnas(ruta2)

# Cantidad de columnas por archivo (ruta1)
print("Columnas por archivo (Measurement complementary)")
for nombre, columnas in inventario1.items():
    print(f"{nombre}: {len(columnas)} columnas")

# Intersección y unión de columnas (ruta1)
listas_de_columnas = list(inventario1.values())
conjuntos_de_columnas = [set(columnas) for columnas in listas_de_columnas]
interseccion = set.intersection(*conjuntos_de_columnas)
union = set.union(*conjuntos_de_columnas)

print(f"\nColumnas en la intersección (comunes a todos): {len(interseccion)}")
print(f"Columnas en la unión (el total combinado): {len(union)}")

print("\nNombres de las columnas en la intersección:")
for columna in sorted(interseccion):
    print(f"- {columna}")

# Columnas que no están en todos los archivos, y en cuáles sí aparecen
columnas_no_comunes = union - interseccion
print(f"\nColumnas que NO están en todos los archivos ({len(columnas_no_comunes)}):")
for c in sorted(columnas_no_comunes):
    archivos_con_esa_col = [n for n, cols in inventario1.items() if c in cols]
    print(f" - '{c}' está en: {archivos_con_esa_col}")

El archivo no se pudo leer trackLog-2022-nov-16_09-44-58.xlsx: 'utf-8' codec can't decode byte 0x9d in position 54: invalid start byte
Columnas por archivo (Measurement complementary)
trackLog-2022-nov.-10_09-23-59 (1).csv: 90 columnas
trackLog-2022-nov.-11_12-47-43.csv: 52 columnas
trackLog-2022-nov.-11_14-46-26.csv: 206 columnas
trackLog-2022-nov-12_20-05-07.csv: 206 columnas
trackLog-2022-nov-13_00-03-52.csv: 206 columnas
trackLog-2022-nov-14_09-03-16.csv: 206 columnas
trackLog-2022-nov-14_13-01-38.csv: 99 columnas
trackLog-2022-nov-16_09-32-26.csv: 99 columnas
trackLog-2022-nov-16_09-44-58.csv: 99 columnas
trackLog-2022-nov.-27_12-43-04.csv: 32 columnas

Columnas en la intersección (comunes a todos): 21
Columnas en la unión (el total combinado): 209

Nombres de las columnas en la intersección:
- Acceleration Sensor(Total)(g)
- Acceleration Sensor(X axis)(g)
- Acceleration Sensor(Y axis)(g)
- Acceleration Sensor(Z axis)(g)
- Altitude
- Bearing
- Device Time
- G(calibrated)
- G(x)
- 

Observación:
1) Indentificación de espacióis en el encabezado de la columna
2) Objetivo: si uno estos 10 archivos en un solo dataframe, ¿qué columnas me conviene conservar?
3) Intersección (20) = las columnas que SÍ están presentes en absolutamente todos los archivos
4) Unión (212) = todas las columnas que existen en al menos un archivo


In [34]:
# Definimos el listado de variables núcleo, junto a variables complementarias
variables_objetivo = [
    'Engine RPM(rpm)',
    'Engine Coolant Temperature(°F)',
    'Intake Air Temperature(°F)',
    'Intake Manifold Pressure(psi)',
    'Absolute Throttle Position B(%)',
    'Engine Load(%)',
    'Voltage (OBD Adapter)(V)',
    'Speed (OBD)(mph)',
    'Acceleration Sensor(Total)(g)',
    'G(x)', 'G(y)', 'G(z)'
]

# Excluimos el archivo con configuración distinta (32 columnas, sin datos de motor)
archivo_excluir = 'trackLog-2022-nov.-27_12-43-04.csv'

inventario1_filtrado = {
    nombre: columnas
    for nombre, columnas in inventario1.items()
    if nombre != archivo_excluir
}

# Recalculamos intersección con los 9 archivos restantes
listas_de_columnas_filtrado = list(inventario1_filtrado.values())
conjuntos_de_columnas_filtrado = [set(columnas) for columnas in listas_de_columnas_filtrado]
interseccion_filtrada = set.intersection(*conjuntos_de_columnas_filtrado)

print(f"Archivos considerados: {len(inventario1_filtrado)} (se excluyó: {archivo_excluir})")
print(f"Columnas en la intersección sin el archivo excluido: {len(interseccion_filtrada)}\n")

# Verificamos cuáles de nuestras variables objetivo sí están disponibles
print("Verificación de variables objetivo:")
for var in variables_objetivo:
    disponible = "ok" if var in interseccion_filtrada else "no"
    print(f"{disponible} {var}")

Archivos considerados: 9 (se excluyó: trackLog-2022-nov.-27_12-43-04.csv)
Columnas en la intersección sin el archivo excluido: 50

Verificación de variables objetivo:
ok Engine RPM(rpm)
ok Engine Coolant Temperature(°F)
ok Intake Air Temperature(°F)
ok Intake Manifold Pressure(psi)
ok Absolute Throttle Position B(%)
ok Engine Load(%)
ok Voltage (OBD Adapter)(V)
ok Speed (OBD)(mph)
ok Acceleration Sensor(Total)(g)
ok G(x)
ok G(y)
ok G(z)
